<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## Baseline Rule

The baseline scoring system identifies webpages that should be reviewed for content refresh.

Each page receives a score based on simple, explainable search performance indicators instead of a machine learning model.

The rule increases the priority score when a page has:

- Declining search trend
- Poor average Google ranking
- Low Click Through Rate (CTR)
- Old content that has not been updated recently
- Falling clicks compared to the previous month

This approach creates a transparent ranking that can be easily understood and audited before training a machine learning model.

### Reason Codes

| Code | Description |
|------|-------------|
| TREND_DOWN | Search traffic is decreasing significantly |
| LOW_POSITION | Average search position is poor |
| LOW_CTR | Click-through rate is below the expected threshold |
| OLD_CONTENT | Content has not been updated for a long period |
| DECLINING_CLICKS | Recent clicks are lower than the previous month |

In [1]:
!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 61), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.87 MiB | 9.41 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/flyrank-ml-internship


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.head())

             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## Building the Ranked Queue

Each webpage receives a baseline score calculated from five independent search signals.

Higher scores indicate that the page is a stronger candidate for content refresh.

The ranking is fully explainable because every point comes from a measurable condition.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

df["baseline_score"] = (
    (df["trend_pct"] < -20).astype(int) * 30 +
    (df["avg_position"] > 20).astype(int) * 20 +
    (df["ctr"] < 0.03).astype(int) * 15 +
    (df["days_since_last_update"] > 180).astype(int) * 20 +
    (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int) * 15
)

df = df.sort_values(
    by="baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df[
    [
        "content_id",
        "baseline_score",
        "trend_pct",
        "avg_position",
        "ctr"
    ]
].head(20))

                 content_id  baseline_score  trend_pct  avg_position    ctr
8860   content_adfc46f3f033              85      -60.0          23.7   0.00
505    content_bfa3d6688324              85      -50.0          21.1   0.00
18652  content_0173fb0dc986              85      -92.1          23.0   0.00
7509   content_7a888d3d99c8              85     -100.0          67.6   0.00
15882  content_129753e3095f              85      -50.0          24.0   0.00
29384  content_f6fdf87348f6              85     -100.0          32.5   0.00
698    content_b16bd7307b39              85      -69.7          31.0   0.00
8458   content_b2b2d6b54dda              85      -50.0          21.0  25.00
3507   content_074ba6ead17b              85      -36.5          48.0   0.00
7021   content_1bfaa38ff26c              85      -74.7          22.2   0.23
16514  content_7368877ea310              85      -81.5          24.8   0.13
11489  content_5feee3994adb              85      -89.1          39.0   0.01
26810  conte

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## Top-20 Review

The highest-ranked pages consistently display multiple negative search performance indicators.

Most pages show:

- Declining traffic trends
- Low CTR
- Poor average search position
- Older content
- Declining clicks compared to the previous month

These pages should be reviewed first because several independent signals suggest they are losing visibility.

The baseline model is transparent and every recommendation can be explained using measurable search metrics.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "trend_pct",
        "avg_position",
        "ctr"
    ]
]


,content_id,baseline_score,trend_pct,avg_position,ctr
8860,content_adfc46f3f033,85,-60.0,23.7,0.00
505,content_bfa3d6688324,85,-50.0,21.1,0.00
18652,content_0173fb0dc986,85,-92.1,23.0,0.00
7509,content_7a888d3d99c8,85,-100.0,67.6,0.00
15882,content_129753e3095f,85,-50.0,24.0,0.00
29384,content_f6fdf87348f6,85,-100.0,32.5,0.00
698,content_b16bd7307b39,85,-69.7,31.0,0.00
8458,content_b2b2d6b54dda,85,-50.0,21.0,25.00
3507,content_074ba6ead17b,85,-36.5,48.0,0.00
7021,content_1bfaa38ff26c,85,-74.7,22.2,0.23


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## Weak Picks

Some pages may receive a high score because of temporary traffic fluctuations, seasonal changes, or external search trends.

These pages should always be manually reviewed before making content changes.

## Leakage Check

The baseline rule only uses historical search performance data.

It does not use any future information, prediction labels, or outcome variables.

Therefore, the ranking is free from target leakage and can safely be used as a transparent baseline before training machine learning models.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")

used_columns = [
    "trend_pct",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "clicks_last_30d",
    "clicks_prev_30d"
]

print("Features used:")
for c in used_columns:
    print("-", c)

print("\nNo future information or labels were used.")

Leakage Check
Features used:
- trend_pct
- avg_position
- ctr
- days_since_last_update
- clicks_last_30d
- clicks_prev_30d

No future information or labels were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.